In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import random
import json
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torchvision

from zennit.composites import EpsilonPlus
from zennit.canonizers import SequentialMergeBatchNorm

from utils.xai_methods import LRP, GradCAMHeatmap

# from utils.xai_methods import GradCAMHeatmap, LRP
from utils.utils import SquareCropAndResize, imagenet_transform, inverse_transform, target_transform


def choose_device() -> str:
    if torch.cuda.is_available():
        return "cuda:0"
    if hasattr(torch.backends, "mps"):
        if torch.backends.mps.is_available():
            return "mps"
    return "cpu"

device = torch.device(choose_device())
print(torch.cuda.get_device_name(device))

# fix seed for reproducibility
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # multi-GPU



In [ ]:
resnet = torchvision.models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V2")
resnet.eval().to(device)

lrp = LRP(
    resnet,
    composite=EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]),
    LRP_target_layer="layer1.0.conv1",
)


pascal_ds = torchvision.datasets.VOCSegmentation(
    root="../data/VOCdevkit",
    year="2012",
    image_set="val",
    download=False,
    transform=imagenet_transform,
    target_transform=SquareCropAndResize(224, interpolation=Image.NEAREST),
)

img_tensor, seg_img = pascal_ds[1353]
img_tensor = img_tensor.unsqueeze(0).to(device)

seg = np.array(seg_img)
seg[seg == 255] = 1
print(np.unique(seg))

mask = np.zeros_like(seg).astype(np.uint8)
mask[seg == 1] = 1

with torch.no_grad():
    pred = resnet(img_tensor).argmax()
heatmap = lrp(img_tensor, pred).cpu()
heatmap = torch.relu(heatmap)
# normalize to [0, 1]
heatmap = heatmap / heatmap.max()
pred_mask = np.zeros_like(heatmap)
pred_mask[heatmap>0.3] = 1
pred_mask = Image.fromarray(pred_mask)
pred_mask = pred_mask.resize((224, 224), Image.NEAREST)
pred_mask = np.array(pred_mask, dtype=np.uint8)



images = [inverse_transform(img_tensor[0].detach().cpu()).permute(1, 2, 0), mask, heatmap, pred_mask]
titles = ['Original Image', 'Mask', 'Heatmap', "Bin heatmap"]
fig, axes = plt.subplots(1, len(images), figsize=(10, 5))

for ax, image, title in zip(axes, images, titles):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()



# Metrics

Read more about metrics:
- https://arxiv.org/abs/2104.05642
- https://arxiv.org/pdf/2206.01653
- https://ieeexplore.ieee.org/abstract/document/7274729
- https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5817231

### Counting metrics

The most frequently used segmentation metrics are counting metrics. In the context of segmentation they are also referred to as overlap metrics, as they essentially measure the overlap between a reference mask and the algorithm prediction (https://arxiv.org/abs/2104.05642)

##### **Intersection over Union (IoU)**

IoU is a metric used to evaluate the accuracy of a predicted segmentation mask against the ground truth mask. It measures the overlap between the predicted and ground truth masks.

$$
\text{IoU} = \frac{|P \cap G|}{|P \cup G|}
$$

Where:
- $|P \cap G|$ is the number of pixels in the intersection of the predicted mask $P$ and the ground truth mask $G$.
- $|P \cup G|$ is the number of pixels in the union of the predicted mask $P$ and the ground truth mask $G$.

The IoU value ranges from 0 to 1, where 1 indicates perfect overlap between the predicted and ground truth masks, and 0 indicates no overlap.

IoU is equivalent to F1 score when used for binary segmentation tasks:

$$
\text{TP} = |P \cap G| \\
\text{FP} = |P - G| \\
\text{FN} = |G - P| \\
$$

$$
\text{F1} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}} = \frac{2 \cdot |P \cap G|}{2 \cdot |P \cap G| + |P - G| + |G - P|} = \frac{2 \cdot |P \cap G|}{|P| + |G|}
$$

In [ ]:
from utils.metrics import IoU

# Initialize IoU metric
iou_metric = IoU()
iou_value = iou_metric(pred_mask, mask)

tp = np.logical_and(pred_mask, mask)
fp = np.logical_and(pred_mask, np.logical_not(mask))
fn = np.logical_and(np.logical_not(pred_mask), mask)



# Manually calculate IoU
# intersection = np.logical_and(pred_mask, mask).sum()
# manual_iou_value = intersection / union

f1 = 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())

union = np.logical_or(pred_mask, mask).sum()
iou = tp.sum() / union

print(f"F1 Score: {f1:.4f}")
print(f"IoU Value (from metric): {iou_value:.4f}")
print(f"Manual IoU Value: {iou:.4f}")


# Visualize the predicted mask, ground truth mask, their intersection, and differences
plt.figure(figsize=(10, 5))

# Create a combined plot
combined_mask = np.ones((*pred_mask.shape, 3))

# Blue for intersection/TP
combined_mask[tp, 0] = 0
combined_mask[tp, 1] = 0
# combined_mask[pred_mask > 0, 1] = 0
# combined_mask[pred_mask > 0, 2] = 0

# Green for ground truth/FN
combined_mask[fn, 0] = 0
combined_mask[fn, 2] = 0


# Red for predicted FP
combined_mask[fp, 1] = 0
combined_mask[fp, 2] = 0

# Plot the combined mask
plt.subplot(1, 2, 1)
plt.imshow(combined_mask)
plt.title('Predicted (Red), Ground Truth (Green)')

red_patch = mpatches.Patch(color='red', label='FP')
green_patch = mpatches.Patch(color='green', label='FN')
white_patch = mpatches.Patch(color='blue', label='TP')
plt.legend(handles=[red_patch, green_patch, white_patch], loc='upper right');

assert np.isclose(iou, iou_value), f"Expected IoU: {iou_value}, but got: {iou}"


##### **Dice Coefficient**

The Dice Coefficient, also known as the Sørensen–Dice index, is a statistical tool used to gauge the similarity between two sets. It is particularly useful in the field of age segmentation, where it measures the overlap between the predicted segmentation and the ground truth.

The Dice Coefficient is defined as:

$$
\text{Dice} = \frac{2 \times |A \cap B|}{|A| + |B|}
$$

where:
- $|A \cap B|$ is the number of elements in the intersection of the predicted set $A$ and the ground truth set $B$.
- $|A|$ and $|B|$ are the number of elements in the predicted set and the ground truth set, respectively.

The value of the Dice Coefficient ranges from 0 to 1, where 0 indicates no overlap between the sets, and 1 indicates perfect overlap.

the Dice Coefficient is the by far most widely used metric in the field of medical image analysis. It is identical to the F1 Score and closely related to the IoU, which is identical to the Jaccard Index:

$$
\text{IoU} = \frac{\text{Dice}}{2 - \text{Dice}}
$$

$$
\text{Dice} = \frac{2 \times \text{IoU}}{1 + \text{IoU}}
$$





In [ ]:
import numpy as np
from utils.metrics import DiceCoefficient

# Manually calculate the Dice Coefficient
intersection = np.logical_and(pred_mask, mask).sum()
manual_dice = 2 * intersection / (pred_mask.sum() + mask.sum())



# Calculate Dice using the implementation from metrics
dice_metric = DiceCoefficient()
metrics_dice = dice_metric(pred_mask, mask)

# Print and compare the results
print(f"Manual Dice Coefficient: {manual_dice}")
print(f"Metrics Dice Coefficient: {metrics_dice}")

import matplotlib.pyplot as plt

# Plot the ground truth and predicted mask as contours
plt.figure(figsize=(8, 8))
plt.contour(mask, levels=[0.5], colors='green', linewidths=2, label='Ground Truth')
plt.contour(pred_mask, levels=[0.5], colors='blue', linewidths=2, label='Predicted Mask')

# Calculate the error mask
error_mask = np.logical_xor(mask, pred_mask)

# Calculate the intersection mask
intersection_mask = np.logical_and(mask, pred_mask)

# Create a 3-channel image for the error mask with red color
error_mask_rgb = np.ones((*error_mask.shape, 3))
error_mask_rgb[error_mask] = [1, 0, 0]  # Red color in RGB

# Add intersection to the error_mask_rgb with a different color (e.g., yellow)
error_mask_rgb[intersection_mask] = [1, 1, 0]  # Yellow color in RGB

# Plot the error mask
plt.imshow(error_mask_rgb)

# Add legend
green_patch = mpatches.Patch(color='green', label='Ground Truth')
blue_patch = mpatches.Patch(color='blue', label='Predicted Mask')
red_patch = mpatches.Patch(color=[1, 0, 0], label='Error')  # Red color in RGB
yellow_patch = mpatches.Patch(color=[1, 1, 0], label='Intersection')  # Yellow color in RGB
plt.legend(handles=[green_patch, blue_patch, red_patch, yellow_patch], loc='upper right')

# Display the plot
plt.title('Contour and Error Visualization')
plt.axis('off')
plt.show()

assert np.isclose(manual_dice, metrics_dice), f"Expected Dice Coefficient: {manual_dice}, but got: {metrics_dice}"

### Distance-based metrics

Overlap-based metrics are often complemented by distance-based metrics that operate exclusively on the TP and compute one or several distances between the reference and the prediction. Apart from a few exceptions, distance-based metrics are often boundary-based metrics which focus on assessing the accuracy of object boundaries (https://arxiv.org/abs/2104.05642)

##### **Boundary IoU (BIoU)**

Boundary IoU (BIoU) measures the Intersection over Union (IoU) of the boundaries (contours) of predicted segmentation masks and ground truth masks. The equation for Boundary IoU can be defined as follows:

$$
BIoU = \frac{\text{Boundary Intersection}}{\text{Boundary Union}} = \frac{\sum_{i=1}^{N} \mathbf{1}(\hat{y}_i^{boundary} == y_i^{boundary})}{\sum_{i=1}^{N} \mathbf{1}(\hat{y}_i^{boundary} \lor y_i^{boundary})}
$$

Where:
- $N$ is the total number of boundary pixels.
- $\hat{y}_i^{boundary}$ is the predicted boundary pixel $i$.
- $y_i^{boundary}$ is the ground truth boundary pixel $i$.
- $\mathbf{1}(\hat{y}_i^{boundary} == y_i^{boundary})$ is an indicator function that equals 1 if the predicted boundary pixel matches the ground truth boundary pixel, and 0 otherwise.
- $\mathbf{1}(\hat{y}_i^{boundary} \lor y_i^{boundary})$ is the union of the boundary regions.


In [ ]:
from utils.metrics import BoundaryIoU, extract_boundary

dilation_radius = 3

predicted_boundary = extract_boundary(pred_mask, dilation_radius)
ground_truth_boundary = extract_boundary(mask, dilation_radius)

# Calculate the intersection and union of the boundaries
boundary_intersection = np.sum(predicted_boundary & ground_truth_boundary)
boundary_union = np.sum(predicted_boundary | ground_truth_boundary)

boundary_iou_value = BoundaryIoU(dilation_radius=dilation_radius)(pred_mask, mask)
print(boundary_intersection / boundary_union)
print(boundary_iou_value)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Plot ground truth and predicted boundaries
ax1.imshow(ground_truth_boundary, cmap='Reds', alpha=0.5)
ax1.imshow(predicted_boundary, cmap='Blues', alpha=0.5)
ax1.set_title('Ground Truth and Predicted Boundaries')
ax1.axis('off')

# Visualize the intersection of the boundaries
intersection_visual = np.ones_like(predicted_boundary, dtype=np.uint8) * 255
intersection_visual[predicted_boundary & ground_truth_boundary] = 0

ax2.imshow(intersection_visual, cmap='gray')
ax2.set_title('Boundary Intersection')
ax2.axis('off')

plt.tight_layout()
plt.show()

# Assertion to check if both methods give the same result
assert np.isclose(boundary_intersection / boundary_union, boundary_iou_value), "The calculated BIoU values do not match!"



##### **Hausdorff Distance (HD)**

The Hausdorff Distance (HD) is a measure of the maximum distance between the boundary points of two binary masks: the predicted segmentation mask and the ground truth mask. It is used to determine the degree of mismatch between the boundaries of the two masks.

The Hausdorff Distance is defined as:

$$
HD(P, G) = \max \left\{ \sup_{x \in \partial P} \inf_{y \in \partial G} d(x, y), \sup_{y \in \partial G} \inf_{x \in \partial P} d(y, x) \right\}
$$

Where:
- $\partial P$ is the boundary of the predicted mask $P$.
- $\partial G$ is the boundary of the ground truth mask $G$.
- $d(x, y)$ is the Euclidean distance between boundary points $x$ and $y$.
- $\sup$ denotes the supremum (maximum) distance.
- $\inf$ denotes the infimum (minimum) distance.

The Hausdorff Distance considers the worst-case scenario by taking the maximum of the minimum distances between the boundary points of the two masks. It is sensitive to outliers and provides a measure of the largest boundary discrepancy between the predicted and ground truth masks.


In [ ]:
from utils.metrics import HausdorffDistance
from scipy.spatial.distance import directed_hausdorff

# Manual calculation of Hausdorff Distance
pred_boundary = np.column_stack(np.where(pred_mask > 0))
gt_boundary = np.column_stack(np.where(mask > 0))

hd_pred_to_gt, hd_pred_index, hd_gt_index = directed_hausdorff(pred_boundary, gt_boundary)
hd_gt_to_pred, hd_gt_index, hd_pred_index = directed_hausdorff(gt_boundary, pred_boundary)

manual_hd = max(hd_pred_to_gt, hd_gt_to_pred)

# Calculate HD using the metric from utils.metrics
hd_metric = HausdorffDistance()
metric_hd = hd_metric(pred_mask, mask)

print(f"Manual HD: {manual_hd:.4f}")
print(f"Metric HD: {metric_hd:.4f}")

# Visualization of Hausdorff Distance
plt.figure(figsize=(6, 6))

# Plot predicted mask boundary
plt.contour(pred_mask, colors='blue', linewidths=0.5)
plt.contour(mask, colors='green', linewidths=0.5)
plt.gca().invert_yaxis()  # Invert y-axis to correct orientation
plt.title('Predicted Mask')

# Find the points that give the Hausdorff distance
if hd_pred_to_gt > hd_gt_to_pred:
    hd_point_pred = pred_boundary[hd_pred_index]
    hd_point_gt = gt_boundary[hd_gt_index]
else:
    hd_point_gt = gt_boundary[hd_gt_index]
    hd_point_pred = pred_boundary[hd_pred_index]

# Plot the Hausdorff distance line
# Plot the Hausdorff distance line
plt.plot([hd_point_pred[1], hd_point_gt[1]], [hd_point_pred[0], hd_point_gt[0]], 'r-', linewidth=2, label=f'Hausdorff Distance')
plt.scatter(hd_point_pred[1], hd_point_pred[0], color='r')
plt.scatter(hd_point_gt[1], hd_point_gt[0], color='r')
# Calculate the length of the line using the distance formula
line_length = np.sqrt((hd_point_gt[1]- hd_point_pred[1])**2 + (hd_point_gt[0] - hd_point_pred[0])**2)
# Add text above the line
mid_x = (hd_point_pred[1] + hd_point_gt[1]) / 2
mid_y = (hd_point_pred[0] + hd_point_gt[0]) / 2
plt.text(mid_x, mid_y-4, f'{line_length:.2f}', color='red', fontweight='bold', 
         ha='center', va='bottom', bbox=dict(facecolor='black', edgecolor='none', alpha=0.1))
plt.legend()

plt.tight_layout()

# Add assertion to check if manual HD and metric HD are close
assert np.isclose(manual_hd, metric_hd, rtol=1e-5), f"Manual HD ({manual_hd:.4f}) and Metric HD ({metric_hd:.4f}) are not close enough"


##### **Average Surface Distance (ASD)**

ASD calculates the average distance between the surface points of two binary masks: the predicted segmentation mask and the ground truth mask. The distance is typically computed based on the Euclidean distance between the nearest boundary points of the two masks.

$$
ASD = \frac{1}{2} \left( \frac{1}{|\partial P|} \sum_{x \in \partial P} d(x, \partial G) + \frac{1}{|\partial G|} \sum_{y \in \partial G} d(y, \partial P) \right)
$$

Where:
- $|\partial P|$ is the number of points on the boundary of the predicted mask $P$.
- $|\partial G|$ is the number of points on the boundary of the ground truth mask $G$.
- $d(x, \partial G)$ is the Euclidean distance from boundary point $x$ of $P$ to the nearest point on the boundary of $G$.
- $d(y, \partial P)$ is the Euclidean distance from boundary point $y$ of $G$ to the nearest point on the boundary of $P$.

The distances are computed for both directions (predicted to ground truth and ground truth to predicted), and the average of these distances is taken.

<!-- ##### **Global Consistency Error (GCE)**

The GCE measures the consistency between two segmentation masks. It calculates how much one segmentation can be viewed as a refinement of the other.

$$
GCE = \frac{1}{N} \sum_{p=1}^{N} \min(E(S_p, G_p), E(G_p, S_p))
$$

Where:
- $N$ is the total number of pixels.
- $S_p$ is the predicted segmentation at pixel $p$.
- $G_p$ is the ground truth segmentation at pixel $p$.
- $E(S_p, G_p)$ is the local refinement error for pixel $p$ when treating the predicted segmentation $S$ as a refinement of the ground truth $G$.

The **local refinement error** $E(S_p, G_p)$ is computed as the number of pixels in the region around pixel $p$ in the predicted segmentation $S$ that do not match the corresponding region in the ground truth segmentation $G$, normalized by the union of the regions. -->


In [ ]:
from utils.metrics import AverageSurfaceDistance
from scipy.ndimage import binary_erosion, distance_transform_edt
from scipy.spatial.distance import cdist

# Initialize Average Surface Distance metric
asd_metric = AverageSurfaceDistance()

# Calculate ASD using the metric
asd_value = asd_metric(pred_mask, mask)

# Manually calculate ASD
pred_boundary = pred_mask - binary_erosion(pred_mask)
gt_boundary = mask - binary_erosion(mask)

pred_boundary_coords = np.column_stack(np.where(pred_boundary > 0))
gt_boundary_coords = np.column_stack(np.where(gt_boundary > 0))


pred_to_gt_distances = distance_transform_edt(1 - gt_boundary)
gt_to_pred_distances = distance_transform_edt(1 - pred_boundary)

pred_to_gt_surface_dist = pred_to_gt_distances[pred_boundary > 0]

gt_to_pred_surface_dist = gt_to_pred_distances[gt_boundary > 0]

manual_asd_value = (np.mean(pred_to_gt_surface_dist) + np.mean(gt_to_pred_surface_dist)) / 2

print(asd_value)
print(manual_asd_value)

# Visualize the predicted boundary and the distances to the ground truth boundary
plt.figure(figsize=(10, 4))

# Visualize the predicted boundary and the distances to the ground truth boundary
plt.subplot(1, 2, 1)
im1 = plt.imshow(pred_to_gt_distances, cmap='hot', interpolation='nearest')
plt.contour(pred_boundary, colors='white', linewidths=0.1, label='Predicted Boundary')
plt.title('Predicted to Ground Truth')

# Visualize the ground truth boundary and the distances to the predicted boundary
plt.subplot(1, 2, 2)
im2 = plt.imshow(gt_to_pred_distances, cmap='hot', interpolation='nearest')
plt.contour(gt_boundary, colors='white', linewidths=0.1, label='Ground Truth Boundary')
plt.title('Ground Truth to Predicted')

# Add a single color bar for both subplots
cbar = plt.colorbar(im2, ax=plt.gcf().get_axes(), label='Distance')

# Visualize the calculation step by step in a single plot with overlay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Overlay boundaries with different colors for masks
ax1.imshow(pred_mask, cmap='Reds', alpha=0.5)
ax1.imshow(mask, cmap='Blues', alpha=0.5)
ax1.scatter(pred_boundary_coords[:, 1], pred_boundary_coords[:, 0], s=1, c='red', label='Predicted Boundary')
ax1.scatter(gt_boundary_coords[:, 1], gt_boundary_coords[:, 0], s=1, c='blue', label='Ground Truth Boundary')

# Overlay distances
for i, (x, y) in enumerate(pred_boundary_coords):
    nearest_gt = gt_boundary_coords[np.argmin(cdist([pred_boundary_coords[i]], gt_boundary_coords))]
    ax1.plot([y, nearest_gt[1]], [x, nearest_gt[0]], 'r-', linewidth=0.5)

for i, (x, y) in enumerate(gt_boundary_coords):
    nearest_pred = pred_boundary_coords[np.argmin(cdist([gt_boundary_coords[i]], pred_boundary_coords))]
    ax1.plot([y, nearest_pred[1]], [x, nearest_pred[0]], 'b-', linewidth=0.5)

ax1.set_title('ASD Calculation Visualization')
ax1.legend()
ax1.axis('off')

# Heat plot reflecting error distance
error_distances = np.zeros(pred_mask.shape)
for i, (x, y) in enumerate(pred_boundary_coords):
    nearest_gt = gt_boundary_coords[np.argmin(cdist([pred_boundary_coords[i]], gt_boundary_coords))]
    error_distances[x, y] = np.linalg.norm([x - nearest_gt[0], y - nearest_gt[1]])

for i, (x, y) in enumerate(gt_boundary_coords):
    nearest_pred = pred_boundary_coords[np.argmin(cdist([gt_boundary_coords[i]], pred_boundary_coords))]
    error_distances[x, y] = np.linalg.norm([x - nearest_pred[0], y - nearest_pred[1]])

ax2.imshow(error_distances, cmap='hot', interpolation='nearest')
cbar = plt.colorbar(ax2.imshow(error_distances, cmap='hot', interpolation='nearest'), ax=ax2, label='Error Distance', shrink=0.8)
ax2.set_title('Error Distance Heatmap')
ax2.axis('off')


# Assert the manually calculated ASD is as expected
assert np.isclose(manual_asd_value, asd_value), f"Expected ASD: {manual_asd_value}, but got: {asd_value}"


##### **Average Symmetric Surface Distance (ASSD)**

The Average Symmetric Surface Distance (ASSD) (also, sometimes referred to as Mean Average Surface Distance (MASD)) is a metric used to evaluate the similarity between two segmentation masks by measuring the average distance between their boundaries. It is particularly useful for assessing the accuracy of predicted segmentation masks against ground truth masks.

The ASSD is calculated as follows:

1. Extract the boundaries of the predicted and ground truth masks.
2. Compute the distance from each point on the predicted boundary to the nearest point on the ground truth boundary.
3. Compute the distance from each point on the ground truth boundary to the nearest point on the predicted boundary.
4. Average these distances to obtain the ASSD.

The formula for ASSD is:

$$
ASSD = \frac{1}{|B_P| + |B_G|} \left( \sum_{p \in B_P} d(p, B_G) + \sum_{g \in B_G} d(g, B_P) \right)
$$

Where:
- $B_P$ is the set of boundary points of the predicted mask.
- $B_G$ is the set of boundary points of the ground truth mask.
- $d(p, B_G)$ is the distance from a point $p$ on the predicted boundary to the nearest point on the ground truth boundary.
- $d(g, B_P)$ is the distance from a point $g$ on the ground truth boundary to the nearest point on the predicted boundary.

$$
d(x, B) = \min_{z \in B} \left\| x - z \right\|
$$


The ASSD value is a non-negative number, where a lower value indicates a better match between the predicted and ground truth boundaries.


In [ ]:
from utils.metrics import AverageSymmetricSurfaceDistance, extract_boundary
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# Initialize Average Symmetric Surface Distance metric
assd_metric = AverageSymmetricSurfaceDistance(dilation_radius=1)

# Manually calculate ASSD
predicted_boundary = extract_boundary(pred_mask, assd_metric.dilation_radius)
ground_truth_boundary = extract_boundary(mask, assd_metric.dilation_radius)

pred_boundary_coords = np.column_stack(np.where(predicted_boundary > 0))
gt_boundary_coords = np.column_stack(np.where(ground_truth_boundary > 0))

if len(pred_boundary_coords) == 0 and len(gt_boundary_coords) == 0:
    manual_assd_value = 0.0
elif len(pred_boundary_coords) == 0 or len(gt_boundary_coords) == 0:
    manual_assd_value = np.inf
else:
    distances_pred_to_gt = cdist(pred_boundary_coords, gt_boundary_coords).min(axis=1)
    distances_gt_to_pred = cdist(gt_boundary_coords, pred_boundary_coords).min(axis=1)
    manual_assd_value = (np.sum(distances_pred_to_gt) + np.sum(distances_gt_to_pred)) / (len(pred_boundary_coords) + len(gt_boundary_coords))


# Visualize the calculation step by step in two separate plots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 7))

# Overlay boundaries with different colors for masks
ax1.contour(pred_mask, colors='blue', linewidths=1, label='Predicted')
ax1.contour(mask, colors='green', linewidths=1, label='Ground Truth')
ax2.contour(pred_mask, colors='blue', linewidths=1, label='Predicted')
ax2.contour(mask, colors='green', linewidths=1, label='Ground Truth')

# Overlay distances from predicted to ground truth
for i, (x, y) in enumerate(pred_boundary_coords):
    nearest_gt = gt_boundary_coords[np.argmin(cdist([pred_boundary_coords[i]], gt_boundary_coords))]
    ax1.plot([y, nearest_gt[1]], [x, nearest_gt[0]], 'r-', linewidth=0.5)

ax1.set_title('Predicted to Ground Truth')
# ax1.legend()
ax1.axis('off')

# Overlay distances from ground truth to predicted
for i, (x, y) in enumerate(gt_boundary_coords):
    nearest_pred = pred_boundary_coords[np.argmin(cdist([gt_boundary_coords[i]], pred_boundary_coords))]
    ax2.plot([y, nearest_pred[1]], [x, nearest_pred[0]], 'r-', linewidth=0.5)

ax2.set_title('Ground Truth to Predicted')
# ax2.legend()
ax2.axis('off')

# Heat plot reflecting error distance
error_distances = np.zeros(pred_mask.shape)
for i, (x, y) in enumerate(pred_boundary_coords):
    nearest_gt = gt_boundary_coords[np.argmin(cdist([pred_boundary_coords[i]], gt_boundary_coords))]
    error_distances[x, y] = np.linalg.norm([x - nearest_gt[0], y - nearest_gt[1]])

for i, (x, y) in enumerate(gt_boundary_coords):
    nearest_pred = pred_boundary_coords[np.argmin(cdist([gt_boundary_coords[i]], pred_boundary_coords))]
    error_distances[x, y] = np.linalg.norm([x - nearest_pred[0], y - nearest_pred[1]])

ax3.imshow(error_distances, cmap='hot', interpolation='nearest')
cbar = plt.colorbar(ax3.imshow(error_distances, cmap='hot', interpolation='nearest'), ax=ax3, label='Error Distance', shrink=0.8)
ax3.set_title('Error Distance Heatmap')
ax3.axis('off')


assd_value = assd_metric(pred_mask, mask)
print(f"ASSD Value: {assd_value}")
print(f"Manual ASSD Value: {manual_assd_value}")

# Assert the manually calculated ASSD is as expected
assert np.isclose(manual_assd_value, assd_value), f"Expected ASSD: {assd_value}, but got: {manual_assd_value}"


##### **Normalized Surface Distance (NSD)**

The Normalized Surface Distance (NSD) is a metric used to evaluate the similarity between two binary masks, typically in the context of image segmentation. It quantifies the average distance between the surface points of the predicted mask and the ground truth mask, normalized by the diagonal length of the bounding box encompassing the union of both masks. This normalization allows the NSD to be scale-invariant, making it a robust measure for comparing segmentation results across different image sizes and resolutions.

The NSD is calculated as follows:

1. **Distance Transform**: Compute the distance transform for both the ground truth mask $T_{GT}$ and the predicted mask $T_{P}$.

2. **Surface Points Extraction**: Identify the surface points of both masks.

3. **Distance Calculation**: Calculate the distances from the surface points of one mask to the other using the distance transforms.

4. **Mean Surface Distance**: Compute the mean of these distances:
   $$
    MSD = \frac{1}{2} \left( \frac{\sum_{s \in S_{GT}} DT_{P}(s)}{|S_{GT}|} + \frac{\sum_{s \in S_{P}} DT_{GT}(s)}{|S_{P}|} \right)
   $$
   where $S_{GT}$ and $S_{P}$ are the sets of surface points for the ground truth and predicted masks, respectively.

5. **Normalization**: Normalize the mean surface distance by the diagonal of the bounding box of the union of the masks:
   $$
    NSD = \frac{MSD}{\sqrt{\sum_{i=1}^{n} (d_i)^2}}
   $$
   where $d_i$ are the dimensions of the bounding box.

A lower NSD value indicates a closer match between the predicted and ground truth boundaries, while a higher value suggests greater disparity.

In [ ]:
from utils.metrics import NormalizedSurfaceDistance

spacing=(1.0, 1.0)

# # Calculate Normalized Surface Distance
nsd_value = NormalizedSurfaceDistance(spacing)(pred_mask, mask)


# Step 1: Calculate the distance transform for the ground truth and predicted masks
dt_mask = distance_transform_edt(~mask, sampling=spacing)
dt_pred_mask = distance_transform_edt(~pred_mask, sampling=spacing)

# Step 2: Extract the surface points of the masks
surface_mask = np.logical_xor(mask, np.logical_and(mask, np.roll(mask, 1, axis=0)))
surface_mask |= np.logical_xor(mask, np.logical_and(mask, np.roll(mask, 1, axis=1)))

surface_pred_mask = np.logical_xor(pred_mask, np.logical_and(pred_mask, np.roll(pred_mask, 1, axis=0)))
surface_pred_mask |= np.logical_xor(pred_mask, np.logical_and(pred_mask, np.roll(pred_mask, 1, axis=1)))

# Step 3: Calculate the distances from the surface points of the ground truth mask to the predicted mask
distances_mask_to_pred = dt_pred_mask[surface_mask]
distances_pred_to_mask = dt_mask[surface_pred_mask]

# Step 4: Calculate the mean surface distance
mean_surface_distance = (np.mean(distances_mask_to_pred) + np.mean(distances_pred_to_mask)) / 2.0

# Step 5: Normalize by the diagonal of the bounding box of the union of the masks
bbox_diagonal = np.sqrt(np.sum((np.array(mask.shape) * np.array(spacing)) ** 2))
nsd = mean_surface_distance / bbox_diagonal

# Print the manually calculated NSD
print(nsd)
print(nsd_value)

# Visualization
plt.figure(figsize=(8, 8))

# Compute distance transforms
dt_mask = distance_transform_edt(~mask)
dt_pred_mask = distance_transform_edt(~pred_mask)

# Extract surface points
surface_mask = np.logical_xor(mask, np.logical_and(mask, np.roll(mask, 1, axis=0)))
surface_mask |= np.logical_xor(mask, np.logical_and(mask, np.roll(mask, 1, axis=1)))

surface_pred_mask = np.logical_xor(pred_mask, np.logical_and(pred_mask, np.roll(pred_mask, 1, axis=0)))
surface_pred_mask |= np.logical_xor(pred_mask, np.logical_and(pred_mask, np.roll(pred_mask, 1, axis=1)))

# Calculate distances from surface points
distances_mask_to_pred = dt_pred_mask[surface_mask]
distances_pred_to_mask = dt_mask[surface_pred_mask]

distance_error = np.zeros_like(mask)
distance_error[surface_mask] = dt_pred_mask[surface_mask]
distance_error[surface_pred_mask] = dt_mask[surface_pred_mask]

# plt.contour(mask, colors='green', linewidths=1)
# plt.contour(pred_mask, colors='blue', linewidths=1)

plt.imshow(distance_error, cmap='hot')
plt.colorbar(label='Distance (pixels)', shrink=0.75)
plt.tight_layout()

assert np.isclose(nsd_value, nsd), f"Expected NSD: {nsd}, but got: {nsd_value}"